# cv_experience_ocr_bert — pipeline en Google Colab

Este notebook solo **orquesta** los scripts de `src/`; toda la lógica vive en el repositorio.

**Antes de empezar**
1. *Runtime → Change runtime type → T4 GPU*.
2. En 🔑 *Secrets* (barra izquierda) crear `HF_TOKEN` con tu token de Hugging Face y activar *Notebook access*.
3. Si el repositorio de GitHub es **privado**, crear también `GITHUB_TOKEN` (token con permiso de lectura del repo).

Los datos, modelos y resultados se guardan en Google Drive (`MyDrive/cv_experience_ocr_bert`), así no se pierden si Colab se reinicia.

In [ ]:
# --- Configuración ---
GITHUB_USER = "BritneyMiranda"   # tu usuario de GitHub
REPO = "cv_experience_ocr_bert"
BRANCH = "main"                  # rama a ejecutar (p. ej. "feature/data-sampling" antes de mergear)
DRIVE_ROOT = "/content/drive/MyDrive/cv_experience_ocr_bert"

In [ ]:
!nvidia-smi

In [ ]:
import os
from google.colab import drive, userdata

drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.environ["CV_STORAGE_DIR"] = DRIVE_ROOT  # src/utils.py guarda data/, models/ y output/ aquí
os.environ["PYTHONUNBUFFERED"] = "1"      # los scripts muestran su progreso en tiempo real

try:
    gh_token = userdata.get("GITHUB_TOKEN")
except Exception:
    gh_token = None
auth = f"{gh_token}@" if gh_token else ""

REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone -q https://{auth}github.com/{GITHUB_USER}/{REPO}.git {REPO_DIR}
!git -C {REPO_DIR} fetch -q origin
!git -C {REPO_DIR} checkout -q {BRANCH}
!git -C {REPO_DIR} pull -q origin {BRANCH}
%cd {REPO_DIR}
!git log --oneline -3

In [ ]:
# Login en Hugging Face con el token guardado en Secrets (nunca escribirlo en el notebook)
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import whoami
print("Conectado a Hugging Face como:", whoami()["name"])

## Fase 1 — Muestreo y descarga (`src/download_data.py`)

1. Se lee un índice liviano (sin imágenes) de todos los shards y se sortean 800 CVs sin foto por ocupación con semilla 42.
2. Se descarga cada shard (barra de progreso), se guardan solo las imágenes sorteadas en Drive y se borra el shard.

Si Colab se desconecta, al volver a ejecutar se saltan las imágenes ya guardadas.

In [ ]:
!python -m src.download_data --inspect

In [ ]:
!python -u -m src.download_data

In [ ]:
import json
import pandas as pd

display(pd.read_csv(f"{DRIVE_ROOT}/output/tables/target_distribution.csv"))
print(json.dumps(json.load(open(f"{DRIVE_ROOT}/output/metrics/sampling_summary.json")), indent=2))

## Fase 2 — PaddleOCR (`src/ocr_extraction.py`)

PaddleOCR se instala en un **entorno aislado con Python 3.12** (`/content/ocr_env`): Colab usa Python 3.13 y sus librerías CUDA de PyTorch podrían chocar con las de PaddlePaddle. Los scripts de OCR se ejecutan con ese Python; todo lo demás, con el de Colab. Esta celda se repite en cada sesión nueva (~3 min).

In [ ]:
OCR_PY = "/content/ocr_env/bin/python"
PADDLE_INDEX = "https://www.paddlepaddle.org.cn/packages/stable/cu126/"

!pip install -q uv
!uv venv -q --python 3.12 /content/ocr_env
!uv pip install -q --python {OCR_PY} paddlepaddle-gpu==3.2.0 --index-url {PADDLE_INDEX} --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match
!uv pip install -q --python {OCR_PY} paddleocr==3.2.0 pandas
!{OCR_PY} -c "import paddle, paddleocr; paddle.utils.run_check(); print('paddle', paddle.__version__, '| paddleocr', paddleocr.__version__)"

In [ ]:
# Prueba con 1 CV antes del lote completo
!{OCR_PY} -u -m src.ocr_extraction --limit 1 --out-dir {DRIVE_ROOT}/data/ocr_test

import glob, json
r = json.load(open(glob.glob(f"{DRIVE_ROOT}/data/ocr_test/*.json")[0]))
print(f"\n{r['cv_id']}: {r['n_lines']} líneas, {r['n_columns']} columna(s), "
      f"confianza media {r['mean_score']}, {r['seconds']} s\n")
print(r["text"][:1500])

### Elegir el modelo de reconocimiento

Comparamos dos modelos de reconocimiento sobre los mismos 40 CVs al azar (el detector es `PP-OCRv5_server_det` en ambos):
- `en_PP-OCRv5_mobile_rec`: modelo ligero entrenado específicamente para inglés.
- `PP-OCRv5_server_rec`: modelo grande multilingüe (chino/inglés/japonés).

Se mide CER/WER contra el texto de referencia (normalizado: minúsculas y espacios colapsados), tiempo por CV y confianza media.

In [ ]:
BENCH = {"en_mobile": "en_PP-OCRv5_mobile_rec", "server": "PP-OCRv5_server_rec"}
for tag, rec in BENCH.items():
    !{OCR_PY} -u -m src.ocr_extraction --limit 40 --rec-model {rec} --out-dir {DRIVE_ROOT}/data/ocr_bench/{tag}

In [ ]:
!pip install -q jiwer
import re
import jiwer
import pandas as pd

ref = pd.read_csv(f"{DRIVE_ROOT}/data/raw/reference_text.csv", dtype=str).set_index("cv_id")["text"]
norm = lambda s: re.sub(r"\s+", " ", str(s).lower()).strip()

rows = []
for tag in BENCH:
    for p in glob.glob(f"{DRIVE_ROOT}/data/ocr_bench/{tag}/*.json"):
        r = json.load(open(p))
        truth, hyp = norm(ref[r["cv_id"]]), norm(r["text"])
        rows.append({"modelo": tag, "CER": jiwer.cer(truth, hyp), "WER": jiwer.wer(truth, hyp),
                     "seg_por_CV": r["seconds"], "confianza": r["mean_score"],
                     "dos_columnas": r["n_columns"] == 2})
bench = pd.DataFrame(rows)
display(bench.groupby("modelo").agg(["mean", "median"]).round(4))

In [ ]:
# Muestra para revisar el orden de lectura: 1 CV por ocupación (imagen + JSON de ambos modelos)
import zipfile
from google.colab import files

first_per_occ = {}
for f in sorted(os.listdir(f"{DRIVE_ROOT}/data/ocr_bench/en_mobile")):
    first_per_occ.setdefault(f.rsplit("_", 1)[0], f)
with zipfile.ZipFile("/content/muestra_ocr.zip", "w") as z:
    for f in first_per_occ.values():
        cv = f[:-5]
        z.write(f"{DRIVE_ROOT}/data/raw/images/{cv}.png", f"images/{cv}.png")
        for tag in BENCH:
            z.write(f"{DRIVE_ROOT}/data/ocr_bench/{tag}/{f}", f"{tag}/{f}")
files.download("/content/muestra_ocr.zip")

### OCR completo (4,000 CVs)

Usar el modelo elegido en el benchmark. Cada CV se guarda en `data/ocr/<cv_id>.json` en cuanto termina: si Colab se desconecta, volver a ejecutar la celda de instalación y esta; continúa donde quedó.

In [ ]:
REC_MODEL = "en_PP-OCRv5_mobile_rec"   # cambiar según el benchmark
!{OCR_PY} -u -m src.ocr_extraction --rec-model {REC_MODEL}
print(json.dumps(json.load(open(f"{DRIVE_ROOT}/output/metrics/ocr_summary.json")), indent=2))

## Sincronizar resultados con el repositorio

Empaqueta los archivos pequeños que sí van a GitHub (IDs sorteados, splits, tablas, métricas y figuras) y los descarga como `resultados.zip`. Las imágenes y los modelos se quedan en Drive.

In [ ]:
from google.colab import files

!cd {DRIVE_ROOT} && rm -f /content/resultados.zip && zip -qr /content/resultados.zip data/raw/sampled_ids.csv data/splits data/processed output -x "*.gitkeep"
files.download("/content/resultados.zip")